##### dbutils.notebook.run()
- Allows you to run a **notebook from another notebook** as a **child job**.
- It is primarily used for `orchestrating complex workflows and pipelines`.

**Parent:** 
- `sends via arguments`

**Child:**
- `reads via widgets`

**Key Rules:**

- `Always pass strings only`
- `Use json.dumps() in parent`
- `Use json.loads() in child`
- `Always create widgets in child`
- `Always return string using exit()`

##### Syntax
      dbutils.notebook.run(path, timeout_seconds, arguments={})

**Parameters**

|     Argument          |   Description        |
|-----------------------|----------------------| 
| **path:**             | The **relative or absolute** workspace path to the notebook. |
| **timeout_seconds:**  | The **maximum time** to **wait** for the run to complete. Setting this to **0 means no timeout**. |
| **arguments:**        | A **dictionary of key-value pairs (strings only)** passed to the **child notebook** as **Databricks Widgets**. |

In [0]:
# -----------------------------
# PARENT NOTEBOOK
# -----------------------------

import json

# 1. Define inputs
source_database_names = {
    "bronze": "bronze_db",
    "silver": "silver_db",
    "gold": "gold_db",
    "manual": "manual_db"
    
}

source_table_names = {
    "source": "orders",
    "sales": "dim_sales_data",
    "detail": "dim_sales_data",
    "fact": "fact_sales_data",
    "key_value": "sales_data_dictionary",
    "currency_path": "currency"
}

read_options_dict = {
    "header": "true",
    "inferSchema": "true"
}

write_options_dict = {
    "mode": "overwrite",
    "mergeSchema":"true"
}

source_col = ["order_id", "cust_id", "cust_manager_id", "cust_sub_manager_id", "amount", "date", "sales_id", "cust_sub_manager_name", "cust_manager"]

In [0]:
# 2. Convert to JSON strings

params = {
    "source_database_names": json.dumps(source_database_names),
    "silver_database_name" : source_database_names["silver"],
    "source_table_names": json.dumps(source_table_names),
    "source_table_name" : source_table_names["source"],
    "sales_table_name" : source_table_names["sales"],
    "read_options_dict": json.dumps(read_options_dict),
    "write_options_dict": json.dumps(write_options_dict),
    "source_col": json.dumps(source_col)
}

# 3. Call child notebook
result = dbutils.notebook.run(
    "/Workspace/Users/enugantisuresh12@gmail.com/@AzureADB/databricks/00_dbutils/dbutils.widgets/2_dbutils.widgets.get/json.loads/39_1_Child Notebook_Receiver",
    timeout_seconds=600,
    arguments=params
)

In [0]:
# 4. Read output
print("Raw Result:", result)

# 5. Convert back to dict
result_dict = json.loads(result)

print("\nStatus:", result_dict["status"])
print("\nMessage:", result_dict["message"])

Raw Result: {"status": "SUCCESS", "message": "Processed:\nDB: silver_db\nTable: orders\nColumns: ['order_id', 'cust_id', 'cust_manager_id', 'cust_sub_manager_id', 'amount', 'date', 'sales_id', 'cust_sub_manager_name', 'cust_manager']"}

Status: SUCCESS

Message: Processed:
DB: silver_db
Table: orders
Columns: ['order_id', 'cust_id', 'cust_manager_id', 'cust_sub_manager_id', 'amount', 'date', 'sales_id', 'cust_sub_manager_name', 'cust_manager']


**timeout_seconds=600 means:**

- The **parent notebook** will **wait a maximum of 600 seconds (10 minutes)** for the **child notebook** to finish execution.

- **dbutils.notebook.run()** executes **another notebook (child notebook)**.
- The **parent notebook pauses and waits**.
- If the **child notebook**:
  - `finishes within 600 seconds → result is returned`
  - `takes longer than 600 seconds → timeout error occurs`
- `No timeout (wait forever)`
  - `timeout_seconds = 0`
  - `Wait indefinitely until notebook completes`

**Case 1:** Child finishes in 2 minutes

- `timeout_seconds = 600`
- `Works fine (2 min < 10 min)`

**Case 2:** Child takes 12 minutes

- `timeout_seconds = 600`
- `Error: java.util.concurrent.TimeoutException`

##### End-to-End Flow

     Parent Notebook
       ↓ (json.dumps)
     Arguments (STRING only)
       ↓
     Child Notebook Widgets
       ↓ (json.loads)
     Python Objects (dict/list)
       ↓
     Processing
       ↓
     json.dumps(output)
       ↓
     dbutils.notebook.exit()
       ↓
     Parent receives STRING
       ↓
     json.loads()